## Import ##

In [1]:
# LOCATION : https://github.com/purnasai/Dino_V2
import torch
from torchvision import models, transforms
import cv2
import os
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from PIL import Image
import pickle
from time import gmtime, strftime

os.environ["XFORMERS_DISABLED"] = "1" # Switch to enable xFormers
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

/media/osero/SamsungSSD/miniconda_files/conda/envs/dinov2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import torch
import torch.nn as nn

DINO_PATH_FINETUNED_DOWNLOADED="/media/osero/SamsungSSD/yedek files/dinov2-1_files/eval/teacher_checkpoint.pth"

def get_dino_finetuned_downloaded():
    # load the original DINOv2 model with the correct architecture and parameters. The positional embedding is too large.
    # load vits or vitg
    model=torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')
    #model=torch.hub.load('facebookresearch/dinov2', 'dinov2_vitg14')
    # load finetuned weights
    pretrained = torch.load(DINO_PATH_FINETUNED_DOWNLOADED, map_location=torch.device('cpu'))
    # make correct state dict for loading
    new_state_dict = {}
    for key, value in pretrained['teacher'].items():
        if 'dino_head' in key:
            print('not used')
        else:
            new_key = key.replace('backbone.', '')
            new_state_dict[new_key] = value
    #change shape of pos_embed, shape depending on vits or vitg
    pos_embed = nn.Parameter(torch.zeros(1, 257, 384))
    #pos_embed = nn.Parameter(torch.zeros(1, 257, 1536))
    model.pos_embed = pos_embed
    # load state dict
    model.load_state_dict(new_state_dict, strict=True)
    return model

In [3]:
# # Load DINO ViT model from torchvision (for example, ViT small or base model trained with DINO)
# import torch.nn as nn

# class VideoClassifierLSTM(nn.Module):
#     def __init__(self, num_classes):
#         super(VideoClassifierLSTM, self).__init__()
#         self.dino_model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14')
#         self.fc = nn.Linear(self.dino_model.embed_dim, num_classes)
#         self.dropout = nn.Dropout(0.1)

#     def forward(self, x):        
#         dino_feature = self.dino_model(x)
#         output = self.dropout(dino_feature)
#         output = self.fc(output)  # Take hidden state of the last LSTM layer
#         return output
    
# video_lassifier_model = VideoClassifierLSTM(num_classes=744)

# video_lassifier_model.load_state_dict(torch.load("/home/osero/Desktop/CMPE/dinov2/classsification/lstm/lstm_results/FINE_TUNED_FACE_B_MODEL2025-01-01_18-05-14_3.pth"))

model = get_dino_finetuned_downloaded()
model.to(device)
model.eval()

# Define transform to match the input size for the model
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

Using cache found in /home/osero/.cache/torch/hub/facebookresearch_dinov2_main
/home/osero/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:45: UserWarning: xFormers is disabled (SwiGLU)
  warnings.warn("xFormers is disabled (SwiGLU)")
/home/osero/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/home/osero/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:29: UserWarning: xFormers is disabled (Attention)
  warnings.warn("xFormers is disabled (Attention)")
/home/osero/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/home/osero/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:35: UserWarning: xFormers is disabled (Block)
  warnings.warn("xFormers is disable

not used
not used
not used
not used
not used
not used
not used
not used


## Functions ##

In [4]:
def extract_video_embedding(image_list):
    """Extracts and averages embeddings for a list of frames."""
    embeddings = []
    with torch.no_grad():
        input_tensor = torch.stack(image_list).to(device)
        features = model(input_tensor)
        features = features.cpu().numpy()
        embeddings = [features[i, :] for i in range(features.shape[0])]

    return embeddings

## Process ##

In [5]:
# Step 4: Process All Videos in the Dataset
# Set paths to your video dataset and labels
video_folder = "/media/osero/SamsungSSD/CMPE_SSD/frame-hand_left-c256" 
video_labels = []  # Populate this with the corresponding labels for each video
video_embeddings = []
process_count = 0
for label_folder in sorted(os.listdir(video_folder)):
    process_count += 1

    full_label_folder = os.path.join(video_folder, label_folder)
    label = int(label_folder)
    print("process_count: ", process_count, ' , label: ', label)
    for sample_folder in sorted(os.listdir(full_label_folder)):
        full_sample_folder = os.path.join(full_label_folder, sample_folder)
        image_list = []
        for image_file in sorted(os.listdir(full_sample_folder)):
            full_image_file = os.path.join(full_sample_folder, image_file)
            image = Image.open(full_image_file)
            image = transform(image)
            image_list.append(image)
        video_embedding = extract_video_embedding(image_list)
        video_embeddings.append(video_embedding)
        video_labels.append(label)
    if(process_count % 100 == 0):
        with open('features_left_hand_small_trained_mixed_saved.pickle', 'wb') as handle:
            pickle.dump((video_embeddings, video_labels), handle, protocol=pickle.HIGHEST_PROTOCOL)

with open('features_left_hand_small_trained_mixed.pickle', 'wb') as handle:
    pickle.dump((video_embeddings, video_labels), handle, protocol=pickle.HIGHEST_PROTOCOL)
abc = 4

process_count:  1  , label:  1
process_count:  2  , label:  2
process_count:  3  , label:  3
process_count:  4  , label:  4
process_count:  5  , label:  5
process_count:  6  , label:  6
process_count:  7  , label:  7
process_count:  8  , label:  8
process_count:  9  , label:  9
process_count:  10  , label:  10
process_count:  11  , label:  11
process_count:  12  , label:  12
process_count:  13  , label:  13
process_count:  14  , label:  14
process_count:  15  , label:  15
process_count:  16  , label:  16
process_count:  17  , label:  17
process_count:  18  , label:  18
process_count:  19  , label:  19
process_count:  20  , label:  20
process_count:  21  , label:  21
process_count:  22  , label:  22
process_count:  23  , label:  23
process_count:  24  , label:  24
process_count:  25  , label:  25
process_count:  26  , label:  26
process_count:  27  , label:  27
process_count:  28  , label:  28
process_count:  29  , label:  29
process_count:  30  , label:  30
process_count:  31  , label:

In [6]:
# Step 4: Process All Videos in the Dataset
# Set paths to your video dataset and labels
video_folder = "/media/osero/SamsungSSD/CMPE_SSD/frame-hand_right-c256" 
video_labels = []  # Populate this with the corresponding labels for each video
video_embeddings = []
process_count = 0
for label_folder in sorted(os.listdir(video_folder)):
    process_count += 1

    full_label_folder = os.path.join(video_folder, label_folder)
    label = int(label_folder)
    print("process_count: ", process_count, ' , label: ', label)
    for sample_folder in sorted(os.listdir(full_label_folder)):
        full_sample_folder = os.path.join(full_label_folder, sample_folder)
        image_list = []
        for image_file in sorted(os.listdir(full_sample_folder)):
            full_image_file = os.path.join(full_sample_folder, image_file)
            image = Image.open(full_image_file)
            image = transform(image)
            image_list.append(image)
        video_embedding = extract_video_embedding(image_list)
        video_embeddings.append(video_embedding)
        video_labels.append(label)
    if(process_count % 100 == 0):
        with open('features_right_hand_small_trained_mixed_saved.pickle', 'wb') as handle:
            pickle.dump((video_embeddings, video_labels), handle, protocol=pickle.HIGHEST_PROTOCOL)

with open('features_right_hand_small_trained_mixed.pickle', 'wb') as handle:
    pickle.dump((video_embeddings, video_labels), handle, protocol=pickle.HIGHEST_PROTOCOL)
abc = 4

process_count:  1  , label:  1
process_count:  2  , label:  2
process_count:  3  , label:  3
process_count:  4  , label:  4
process_count:  5  , label:  5
process_count:  6  , label:  6
process_count:  7  , label:  7
process_count:  8  , label:  8
process_count:  9  , label:  9
process_count:  10  , label:  10
process_count:  11  , label:  11
process_count:  12  , label:  12
process_count:  13  , label:  13
process_count:  14  , label:  14
process_count:  15  , label:  15
process_count:  16  , label:  16
process_count:  17  , label:  17
process_count:  18  , label:  18
process_count:  19  , label:  19
process_count:  20  , label:  20
process_count:  21  , label:  21
process_count:  22  , label:  22
process_count:  23  , label:  23
process_count:  24  , label:  24
process_count:  25  , label:  25
process_count:  26  , label:  26
process_count:  27  , label:  27
process_count:  28  , label:  28
process_count:  29  , label:  29
process_count:  30  , label:  30
process_count:  31  , label:

## Evaluation ##

In [7]:
# with open('features_face_frames_small.pickle', 'wb') as handle:
#     pickle.dump((video_embeddings, video_labels), handle, protocol=pickle.HIGHEST_PROTOCOL)

In [8]:
pickle_file11 = open('features_face_frames_large.pickle', 'rb')
paths11, features11,labels11 = pickle.load(pickle_file11)
cc = 5

FileNotFoundError: [Errno 2] No such file or directory: 'features_face_frames_large.pickle'